<a href="https://colab.research.google.com/github/cristel-est/cat-vs-dog-classifier/blob/main/cats_and_dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub tensorflow opencv-python scikit-learn matplotlib numpy

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")

print("Dataset path:", dataset_path)

100%|██████████| 360M/360M [00:02<00:00, 130MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3


In [ ]:
import os

BASE_PATH = os.path.join(dataset_path, "animals")
CAT_PATH = os.path.join(BASE_PATH, "cat")
DOG_PATH = os.path.join(BASE_PATH, "dog")

print(CAT_PATH)
print(DOG_PATH)

import cv2
import numpy as np

IMG_SIZE = 128

data = []
labels = []

# Cats = 0
for img_name in os.listdir(CAT_PATH):
    img_path = os.path.join(CAT_PATH, img_name)
    img = cv2.imread(img_path)

    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        data.append(img)
        labels.append(0)

# Dogs = 1
for img_name in os.listdir(DOG_PATH):
    img_path = os.path.join(DOG_PATH, img_name)
    img = cv2.imread(img_path)

    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        data.append(img)
        labels.append(1)

data = np.array(data, dtype="float32") / 255.0
labels = np.array(labels)

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

loss, acc = model.evaluate(X_test, y_test)
print("Test accuracy:", acc)

model.save("cat_dog_model.h5")

test_img = os.path.join(CAT_PATH, os.listdir(CAT_PATH)[0])
print("Testing image:", test_img)

import numpy as np
import cv2
import tensorflow as tf

model = tf.keras.models.load_model("cat_dog_model.h5")

img = cv2.imread(test_img)
img = cv2.resize(img, (128,128))
img = img / 255.0
img = np.expand_dims(img, axis=0)


from google.colab import files

# Upload image from your computer
uploaded = files.upload()

# Get uploaded file name
for filename in uploaded.keys():
    print("Uploaded file:", filename)

    # Read and preprocess image (same as your test pipeline)
    img = cv2.imread(filename)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    pred = model.predict(img)

    print("Prediction:", "Dog 🐶" if pred[0][0] > 0.5 else "Cat 🐱")

/root/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals/cat
/root/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals/dog
Data shape: (1000, 128, 128, 3)
Labels shape: (1000,)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 81ms/step - accuracy: 0.5425 - loss: 0.7470 - val_accuracy: 0.6650 - val_loss: 0.6600
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7325 - loss: 0.5586 - val_accuracy: 0.7900 - val_loss: 0.4554
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8313 - loss: 0.4064 - val_accuracy: 0.7950 - val_loss: 0.4487
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.8975 - loss: 0.2621 - val_accuracy: 0.8900 - val_loss: 0.2796
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9438 - loss: 0.1529 - val_accuracy: 0.9100 - val_loss: 0.2096
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9100 - loss: 0.2096 


Test accuracy: 0.9100000262260437
Testing image: /root/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3/animals/cat/00219-200124529.png


In [ ]:
!pip install streamlit tensorflow opencv-python scikit-learn numpy

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import tensorflow as tf
import os

IMG_SIZE = 128
MODEL_PATH = "cat_dog_model.h5"

# ---------------- UI ----------------
st.title("🐱🐶 Cat vs Dog Classifier")
st.write("Upload images and get instant predictions.")

# ---------------- LOAD MODEL ----------------
@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        return None
    return tf.keras.models.load_model(MODEL_PATH)

model = load_model()

if model is None:
    st.error("❌ No trained model found!")
    st.info("👉 Train your model in Colab first and save it as 'cat_dog_model.h5'")
    st.stop()

st.success("✅ Model loaded successfully!")

# ---------------- IMAGE UPLOAD ONLY ----------------
uploaded_files = st.file_uploader(
    "Upload images",
    type=["jpg", "png", "jpeg"],
    accept_multiple_files=True
)

# ---------------- PREDICTION ----------------
if uploaded_files:
    for file in uploaded_files:
        img_bytes = np.frombuffer(file.read(), np.uint8)
        img = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)

        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img_norm = img_resized / 255.0
        img_input = np.expand_dims(img_norm, axis=0)

        pred = model.predict(img_input, verbose=0)[0][0]
        label = "Dog 🐶" if pred > 0.5 else "Cat 🐱"

        st.image(img, caption=f"{file.name} → {label} ({pred:.2f})", width=250)

Writing app.py


In [ ]:
!pip install streamlit pyngrok


from pyngrok import ngrok
ngrok.set_auth_token("3F9MwhKHS1T65ErQPV8nZOtIf0k_2q7wzRkBGyHnQAn1FG4d")


# 1. start streamlit FIRST
!streamlit run app.py &>/content/logs.txt &


# 2. then create tunnel
public_url = ngrok.connect(8501)
print("Open this URL:", public_url)

Open this URL: NgrokTunnel: "https://conducive-rigor-customer.ngrok-free.dev" -> "http://localhost:8501"
